In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=DYbo2p0SKCxjroazlubH6kVpMGwIH4&access_type=offline&code_challenge=hwjDl63yliB61dFOwXjAJTYeHhXoErl2wV7WUHdikfk&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning

In [2]:
from gentropy.dataset.study_locus import StudyLocus

In [3]:
x=StudyLocus.from_parquet(session,"gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus_alzheimer")

In [5]:
x.df.select("studyLocusId").show()

+--------------------+
|        studyLocusId|
+--------------------+
| 6109438569946056978|
| 6388992474978589194|
| 4154989118717074566|
|-7673925720032212461|
| 6360456299763482946|
|-6742466305250328444|
|-6168361428432361140|
| 1916491992423016132|
|-3612515273077152914|
|-8288099943480320096|
| 6814727764900576662|
| 2576050778682258213|
| 6796883077590987845|
|-4585712009512019667|
| 8895835730818824947|
| 7171154626284587162|
|-3402465258691108958|
|-8640267085448358001|
| 3318332793803757311|
|  760299597568413738|
+--------------------+
only showing top 20 rows



In [1]:
from gentropy.common.session import Session
from gentropy.susie_finemapper import SusieFineMapperStep
import os
import hail as hl

path_study_locus="gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus_alzheimer"
path_si="gs://gwas_catalog_data/study_index"
path_out="gs://genetics-portal-dev-analysis/yt4/tmp_to_delete/"

hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={"spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g"})
hl.init(sc=session.spark.sparkContext, log="/dev/null")

L=SusieFineMapperStep(session=session, 
    study_locus_to_finemap="6388992474978589194", 
    study_locus_collected_path=path_study_locus,
    study_index_path=path_si,
    output_path=path_out, 
    locus_radius=100_000,
    locus_L=10)

Loading BokehJS ...

24/04/23 13:43:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null
2024-04-23 13:43:40.342 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'age_distribution' -> 'age_distribution_1'
    'freq_meta' -> 'freq_meta_1'
    'faf_index_dict' -> 'faf_index_dict_1'
    'popmax_index_d

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55668)
Traceback (most recent call last):
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/yt4/Library/Caches/pypoetry/virtualenvs/gentropy-krNFZEZg-py3.10/lib/python3.10/site-packages/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/Users/yt4/Library/Caches/pypoetry/virtualenvs/gentropy-krNFZEZg-py3.1

In [1]:
import numpy as np
from typing import Any
import os
import hail as hl
import pyspark.sql.functions as f
import pandas as pd

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.l2g_prediction import L2GPrediction


Loading BokehJS ...

24/04/23 13:32:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [2]:
from gentropy.susie_finemapper import SusieFineMapperStep

In [3]:
path_study_locus="gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus_alzheimer"
path_gwas1="gs://gwas_catalog_data/harmonised_summary_statistics/GCST90012877.parquet"
path_si="gs://gwas_catalog_data/study_index"
gwas1 = SummaryStatistics.from_parquet(session, path_gwas1)
study_index = StudyIndex.from_parquet(session, path_si)
study_locus=StudyLocus.from_parquet(session=session,path=path_study_locus)

In [4]:
L=SusieFineMapperStep(session=session, 
    study_locus_to_finemap="6388992474978589194", 
    study_locus_collected_path=path_study_locus,
    study_index_path=path_si,
    output_path="gs://genetics-portal-dev-analysis/yt4/", 
    locus_radius=100_000,
    locus_L=10)

2024-04-23 13:32:45.336 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'age_index_dict' -> 'age_index_dict_1'
    'rf' -> 'rf_1'
    'faf_index_dict' -> 'faf_index_dict_1'
    'freq_meta' -> 'freq_meta_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
    'age_distribution' -> 'age_distribution_1'
    'freq_index_dict' -> 'freq_index_dict_1'
2024-04-23 13:32:54.346 Hail: INFO: Coerced sorted dataset
2024-04-23 13:32:58.984 Hail: INFO: Coerced sorted dataset


In [20]:
study_locus=study_locus.annotate_locus_statistics(summary_statistics=gwas1,collect_locus_distance=100000)

In [25]:
study_locus.df.show(32)

+--------------------+----------------+----------+---------+------+------------+----------------+--------------+--------------+-------------------------------+----------------+-------------------+---------------+-----------------+----------------+------------------+----------+-----+--------------------+
|        studyLocusId|       variantId|chromosome| position|region|     studyId|            beta|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|   standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|sampleSize|ldSet|               locus|
+--------------------+----------------+----------+---------+------+------------+----------------+--------------+--------------+-------------------------------+----------------+-------------------+---------------+-----------------+----------------+------------------+----------+-----+--------------------+
| 6109438569946056978|  19_1050875_A_G|        19|  1050875|  null|GCST90012877|-0.07

In [31]:
from pyspark.sql.functions import size
slt_df=study_locus.df

slt_df = slt_df.withColumn("locus_size", size(slt_df.locus))

#slt_df.show(32)
slt_df.show(1,False,True)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
#study_locus.df.write.mode(session.write_mode).parquet("gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus_alzheimer_v2")
study_locus.df.write.parquet("gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus_alzheimer",mode="overwrite")

In [ ]:
x=SusieFineMapperStep(session=session,)

In [ ]:
x=StudyLocus.from_parquet(session=session,path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_ldb_clumping_v3")
x.df.count()


In [ ]:
df=x.df
df=df.filter(f.col("studyID")=="UKB_PPP_EUR_AAMDC_Q9H7C9_OID30236_v1")
df = df.filter(~((f.col("chromosome") == "6") & (f.col("position").between(28000000, 34000000))))
df.count()

x.df=df

In [ ]:
path_sumstats="gs://ukb_ppp_eur_data/summary_stats_2/chromosome=11/"
L=SummaryStatistics.from_parquet(session=session,path=path_sumstats)
L.df=L.df.filter(f.col("studyID")=="UKB_PPP_EUR_AAMDC_Q9H7C9_OID30236_v1")
L.df.write.parquet("gs://genetics-portal-dev-analysis/yt4/toy_gwas")




In [ ]:
L=SummaryStatistics.from_parquet(session=session,path="gs://genetics-portal-dev-analysis/yt4/toy_gwas")

In [ ]:
wbc=x.annotate_locus_statistics(summary_statistics=L,collect_locus_distance=100000)
x.df.write.parquet("gs://genetics-portal-dev-analysis/yt4/toy_studdy_locus")

In [ ]:
unique_values = x.df.select("qualityControls").distinct().show(truncate=False)
print(unique_values)

In [ ]:
from pyspark.sql.functions import col
df=x.df
df = df.filter(~((col("chromosome") == "6") & (col("position").between(28000000, 34000000))))
df.count()




In [ ]:
df.write.parquet("gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_ldb_clumping_v3_no_mhc")

In [ ]:
x.df.show()

In [ ]:
# Clumping of UKBB-PPP
import sys
sys.path.append('/home/yt4/gentropy')

from gentropy.common.session import Session

from gentropy.method.window_based_clumping import WindowBasedClumping
from gentropy.dataset.summary_statistics import SummaryStatistics
from pyspark.sql.functions import col

session = Session()

In [ ]:
path_sumstats="gs://ukb_ppp_eur_data/summary_stats_2"
L=SummaryStatistics.from_parquet(session=session,path=path_sumstats)

wbc=WindowBasedClumping.clump(summary_statistics=L,distance=500000,gwas_significance=1e-15).annotate_locus_statistics(summary_statistics=L,collect_locus_distance=1000000)



summary_statistics=L
distance=500000
gwas_significance=5e-8

df.write.mode(session.write_mode).parquet("gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_wb_clumping")

In [ ]:
wbc.df.show(1)

In [ ]:
from gentropy.dataset.ld_index import LDIndex
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
study_index_path="gs://ukb_ppp_eur_data/study_index"
#ld_index_path="gs://genetics_etl_python_playground/output/python_etl/parquet/XX.XX/ld_index/"
ld_index_path="gs://genetics_etl_python_playground/static_assets/ld_index",
x=StudyLocus.from_parquet(session=session,path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_wb_clumping")

ld_index = LDIndex.from_parquet(session, ld_index_path)
study_index = StudyIndex.from_parquet(session, study_index_path)

from pyspark.sql.functions import array, lit, struct

study_index_df=study_index.df
study_index.df = study_index_df.withColumn("ldPopulationStructure", array(struct(lit("eur").alias("ldPopulation"), lit(1.0).alias("relativeSampleSize"))))


df=x.annotate_ld(study_index, ld_index).clump()

In [ ]:
clumped_study_locus_output_path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_ldb_clumping_v3"
df.df.write.mode(session.write_mode).parquet(clumped_study_locus_output_path)

In [ ]:
df1=StudyLocus.from_parquet(session=session,path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_ldb_clumping_v3")

In [ ]:

L=WindowBasedClumpingStep(session=session,
summary_statistics_input_path="gs://ukb_ppp_eur_data/summary_stats",
study_locus_output_path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_wb_clumping",
distance=500000,
collect_locus=False
)

In [ ]:
from gentropy.ld_based_clumping import LDBasedClumpingStep
L2=LDBasedClumpingStep(session=session,study_locus_input_path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_wb_clumping",
study_index_path="gs://ukb_ppp_eur_data/study_index",
ld_index_path="gs://genetics_etl_python_playground/static_assets/ld_index",
clumped_study_locus_output_path="gs://genetics-portal-dev-analysis/yt4/ukbb_ppp_ldb_clumping")

In [ ]:
path_to_l2g="gs://genetics_etl_python_playground/releases/24.03/locus_to_gene_predictions"
l2g=session.read_parquet(path_to_l2g,schema=L2GPrediction.get_schema())

path_to_cs=["gs://genetics_etl_python_playground/releases/24.03/credible_set/gwas_catalog_PICSed_curated_associations/",
            "gs://genetics_etl_python_playground/releases/24.03/credible_set/gwas_catalog_PICSed_summary_statistics",
            "gs://genetics_etl_python_playground/releases/24.03/credible_set/finngen_susie"]
cred_sets=StudyLocus.from_parquet(session=session,path=path_to_cs)

genes=session.spark.read.parquet("gs://genetics_etl_python_playground/static_assets/targets").select("id","approvedSymbol")
genes=genes.withColumnRenamed("id","geneId")

path_to_study_index1="gs://genetics-portal-dev-analysis/yt4/study_index_finngen_with_efo"
path_to_study_index2="gs://genetics_etl_python_playground/releases/24.03/study_index/gwas_catalog"
si=StudyIndex.from_parquet(session=session, path=[path_to_study_index1,path_to_study_index2])

# Parsing study index

In [ ]:
path_si="gs://ukb_ppp_eur_data/study_index"
si=StudyIndex.from_parquet(session=session, path=[path_si])
si.df.count()

In [ ]:
si.df.select("nSamples").describe().show()

In [ ]:
path_si="gs://genetics_etl_python_playground/releases/24.03/study_index/eqtl_catalogue/study_index"
si=StudyIndex.from_parquet(session=session, path=[path_si])
si.df.count()

In [ ]:
si_tmp=si.df.filter(f.col("studyType")=="eqtl")
si_tmp.count()

In [ ]:
si_tmp.show(1)

In [ ]:
path_si="gs://genetics-portal-dev-analysis/yt4/study_index_finngen_with_efo"
si=StudyIndex.from_parquet(session=session, path=[path_si])
si.df.count()

In [ ]:
cred_sets_df=cred_sets.df
cred_sets_df_susie=cred_sets_df.filter(f.col("fineMappingMethod")=="SuSie")
cred_sets_df_susie.count()

In [ ]:
unique_studyId_count = cred_sets_df_susie.select("studyId").distinct().count()
print(unique_studyId_count)

In [ ]:
path_to_cs=[ "gs://genetics_etl_python_playground/releases/24.03/credible_set/gwas_catalog_PICSed_summary_statistics"]
cred_sets=StudyLocus.from_parquet(session=session,path=path_to_cs)

path_to_study_index2="gs://genetics_etl_python_playground/releases/24.03/study_index/gwas_catalog"
si=StudyIndex.from_parquet(session=session, path=[path_to_study_index2])

In [ ]:
si.df.show(1)

In [ ]:
si_df=si.df
si_df=si_df.filter(f.col("hasSumstats")==True)
si_df.count()

In [ ]:
unique_studyId_count = si_df.select("traitFromSourceMappedIds").distinct().count()
print(unique_studyId_count)

In [ ]:
cred_sets_df = cred_sets.df.join(si_df, on="studyId", how="inner")
cred_sets_df=cred_sets_df.filter(f.col("variantId").isNotNull())
cred_sets_df.count()

In [ ]:
unique_studyId_count = cred_sets_df.select("studyId").distinct().count()
print(unique_studyId_count)

In [ ]:
#genes=session.spark.read.parquet("gs://genetics_etl_python_playground/static_assets/targets")
#genes=genes.withColumnRenamed("id","geneId")
#genes = (genes.withColumn("chromosome", genes["genomicLocation.chromosome"])
#.withColumn("start", genes["genomicLocation.start"])
#.withColumn("strand", genes["genomicLocation.strand"]))
#.withColumn("end", genes["genomicLocation.end"])
#genes.toPandas().to_csv("/Users/yt4/Projects/ot_data/24_03_ppp/genes_annot.tsv", sep="\t", index=False)
#genes=genes.select("geneId","approvedSymbol","chromosome","start","end","strand")

In [ ]:
num_unique_entries = l2g.select("geneId").distinct().count()
print(num_unique_entries)

In [ ]:
cred_sets.df.show(3)

In [ ]:
si.df.count()

In [ ]:
si.df.printSchema()

# IBD

In [ ]:
EFOs=[
    "EFO_0003767", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0003767
    "EFO_0000384", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0000384
    "EFO_0000729", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0000729
    "EFO_0000555", #https://www.ebi.ac.uk/gwas/efotraits/EFO_0000555 
]

In [ ]:
df=si.df
from pyspark.sql.functions import expr

EFOs_str = ",".join([f"'{efo}'" for efo in EFOs])
df = df.filter(expr(f"exists(traitFromSourceMappedIds, x -> x in ({EFOs_str}))"))

df = df.filter(f.col("backgroundTraitFromSourceMappedIds").isNull())

In [ ]:
df.count()

In [ ]:
df.show()

In [ ]:
list_of_studies = df.select("studyId").rdd.flatMap(lambda x: x).collect()

In [ ]:
list_of_studies=["GCST004131"]

In [ ]:
from pyspark.sql.functions import log10

filtered_df = cred_sets.df.filter(cred_sets.df.studyId.isin(list_of_studies))
filtered_df=filtered_df.filter(f.col("variantId").isNotNull())
#filtered_df=filtered_df.filter(f.col("finemappingMethod")=="SuSie")

filtered_df=filtered_df.withColumn("log_mantissa", log10("pValueMantissa"))
filtered_df=filtered_df.withColumn("logpval",f.col("log_mantissa") + f.col("pValueExponent"))

filtered_df=filtered_df.filter(f.col("logpval")<=-8)

In [ ]:
filtered_df.show(3)

In [ ]:
filtered_df.count()

In [ ]:
l2g.show(2)

In [ ]:
merged_df = filtered_df.join(l2g, on="studyLocusId", how="inner")
merged_df.count()

In [ ]:
#merged_df=merged_df.filter(f.col("score")>=0.05)

In [ ]:
merged_df=merged_df.join(genes, on="geneId", how="inner")
merged_df=merged_df.drop("ldSet","locus")

In [ ]:
merged_df.show(2)

In [ ]:
merged_df.toPandas().to_csv("/Users/yt4/Projects/ot_data/24_03_ppp/ibd_l2g.tsv", sep="\t", index=False)

# CAD

In [ ]:
EFOs=[
"EFO_0000319",
"EFO_0000275",
"HP_0002140",
"EFO_0000537",
"EFO_0004269"
"EFO_0000612",
"MONDO_0019171"
"EFO_0003777"
]

In [ ]:
df=si.df
from pyspark.sql.functions import expr

EFOs_str = ",".join([f"'{efo}'" for efo in EFOs])
df = df.filter(expr(f"exists(traitFromSourceMappedIds, x -> x in ({EFOs_str}))"))

df = df.filter(f.col("backgroundTraitFromSourceMappedIds").isNull())

list_of_studies = df.select("studyId").rdd.flatMap(lambda x: x).collect()

len(list_of_studies)

In [ ]:
from pyspark.sql.functions import log10

filtered_df = cred_sets.df.filter(cred_sets.df.studyId.isin(list_of_studies))
filtered_df=filtered_df.filter(f.col("variantId").isNotNull())
#filtered_df=filtered_df.filter(f.col("finemappingMethod")=="SuSie")

filtered_df=filtered_df.withColumn("log_mantissa", log10("pValueMantissa"))
filtered_df=filtered_df.withColumn("logpval",f.col("log_mantissa") + f.col("pValueExponent"))

filtered_df=filtered_df.filter(f.col("logpval")<=-8)

merged_df = filtered_df.join(l2g, on="studyLocusId", how="inner")
merged_df.count()

In [ ]:
merged_df=merged_df.join(genes, on="geneId", how="inner")
merged_df=merged_df.drop("ldSet","locus")
merged_df.show(2)

In [ ]:
merged_df.toPandas().to_csv("/Users/yt4/Projects/ot_data/24_03_ppp/cad_l2g_full.tsv", sep="\t", index=False)

# FEATURES

In [ ]:
from gentropy.dataset.v2g import V2G
from gentropy.dataset.l2g_feature_matrix import L2GFeatureMatrix
from gentropy.dataset.colocalisation import Colocalisation


coloc=Colocalisation.from_parquet(session, "gs://genetics_etl_python_playground/releases/24.03/colocalisation")

v2g = V2G.from_parquet(session, "gs://genetics_etl_python_playground/releases/24.03/variant_to_gene")
#v2g.df.count()

features_list=[
            # average distance of all tagging variants to gene TSS
            "distanceTssMean",
            # minimum distance of all tagging variants to gene TSS
            "distanceTssMinimum",
            # maximum vep consequence score of the locus 95% credible set among all genes in the vicinity
            "vepMaximumNeighborhood",
            # maximum vep consequence score of the locus 95% credible set split by gene
            "vepMaximum",
            # mean vep consequence score of the locus 95% credible set among all genes in the vicinity
            "vepMeanNeighborhood",
            # mean vep consequence score of the locus 95% credible set split by gene
            "vepMean",
            # max clpp for each (study, locus, gene) aggregating over all eQTLs
            "eqtlColocClppMaximum",
            # max clpp for each (study, locus) aggregating over all eQTLs
            "eqtlColocClppMaximumNeighborhood",
            # max clpp for each (study, locus, gene) aggregating over all pQTLs
            "pqtlColocClppMaximum",
            # max clpp for each (study, locus) aggregating over all pQTLs
            "pqtlColocClppMaximumNeighborhood",
            # max clpp for each (study, locus, gene) aggregating over all sQTLs
            "sqtlColocClppMaximum",
            # max clpp for each (study, locus) aggregating over all sQTLs
            "sqtlColocClppMaximumNeighborhood",
            # max clpp for each (study, locus) aggregating over all tuQTLs
            "tuqtlColocClppMaximum",
            # max clpp for each (study, locus, gene) aggregating over all tuQTLs
            "tuqtlColocClppMaximumNeighborhood",
            # # max log-likelihood ratio value for each (study, locus, gene) aggregating over all eQTLs
            # "eqtlColocLlrLocalMaximum",
            # # max log-likelihood ratio value for each (study, locus) aggregating over all eQTLs
            # "eqtlColocLlpMaximumNeighborhood",
            # # max log-likelihood ratio value for each (study, locus, gene) aggregating over all pQTLs
            # "pqtlColocLlrLocalMaximum",
            # # max log-likelihood ratio value for each (study, locus) aggregating over all pQTLs
            # "pqtlColocLlpMaximumNeighborhood",
            # # max log-likelihood ratio value for each (study, locus, gene) aggregating over all sQTLs
            # "sqtlColocLlrLocalMaximum",
            # # max log-likelihood ratio value for each (study, locus) aggregating over all sQTLs
            # "sqtlColocLlpMaximumNeighborhood",
        ]

fm = L2GFeatureMatrix.generate_features(
    features_list=features_list,
    credible_set=cred_sets,
    study_index=si,
    variant_gene=v2g,
    colocalisation=coloc,
).fill_na()